# step_combined

> Phase 2 combined step renderer: dual-column layout for Segment & Align

In [ ]:
#| default_exp components.step_renderer

In [ ]:
#| export
from typing import Any, List

from fasthtml.common import Div, H2, P, Span, Input, Button, Script

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui, ring_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, uppercase, tracking
from cjm_fasthtml_tailwind.utilities.layout import overflow, position, inset, display_tw
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.effects import opacity, ring
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow, flex_wrap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Interaction context
from cjm_fasthtml_interactions.core.context import InteractionContext

# Card stack library
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.components.states import render_loading_state
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

# Local imports
# HTML IDs (page-specific)
from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segmentation.html_ids import SegmentationHtmlIds
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_vad_align.models import VADChunk, AlignmentUrls

# State extraction helpers (combined-level)
from cjm_transcript_segment_align.components.helpers import (
    extract_seg_state, extract_alignment_state,
)

# Decomposition composable renderers
from cjm_transcript_segmentation.components.step_renderer import (
    render_seg_column_body, render_toolbar, render_seg_stats,
    render_seg_footer_content, render_seg_mini_stats_text,
)

# Alignment composable renderers
from cjm_transcript_vad_align.components.step_renderer import (
    render_align_column_body, render_align_mini_stats_text,
    render_align_footer_content,
)
from cjm_transcript_vad_align.components.card_stack_config import (
    ALIGN_CS_IDS,
)

# Shared keyboard config (combined-level)
from cjm_transcript_segment_align.components.keyboard_config import (
    build_combined_kb_system, generate_zone_change_js,
    SWITCH_CHROME_BTN_ID,
)

# Keyboard hints modal
from cjm_fasthtml_keyboard_navigation.components.hints_modal import render_keyboard_hints_modal
from cjm_transcript_segmentation.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS,
)

# Sync controls
from cjm_transcript_segment_align.components.sync_controls import (
    build_extra_actions, generate_sync_script,
    generate_should_play_js, SHOULD_PLAY_FN,
)

# Debug flag for combined step rendering tracing (set False in production)
DEBUG_COMBINED_RENDER = True


## Column Headers with Mini-Stats

Each column has a header showing its title and a badge area for compact stats
that remain visible regardless of which column is active.

In [ ]:
#| export
def _render_column_header(
    title:str,  # Column title (e.g., "Text Decomposition")
    stats_id:str,  # HTML ID for the mini-stats badge area
    header_id:str,  # HTML ID for the column header container
    initial_text:str="--",  # Initial text for the mini-stats badge
) -> Any:  # Column header component
    """Render a column header with title and mini-stats badge."""
    return Div(
        Span(
            title,
            cls=combine_classes(
                font_size.sm, font_weight.bold,
                uppercase, tracking.wide,
                text_dui.base_content.opacity(50)
            )
        ),
        Span(
            initial_text,
            id=stats_id,
            cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm)
        ),
        id=header_id,
        cls=combine_classes(
            flex_display, justify.between, items.center,
            p(3), bg_dui.base_200,
            border_dui.base_300, border.b()
        )
    )

## Mini-Stats Badge OOB

Renders the full decomposition mini-stats badge for OOB swaps from handlers.
Matches the badge styling used in `_render_column_header`.

In [ ]:
#| export
def render_seg_mini_stats_badge(
    segments:List[TextSegment],  # Current segments
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    """Render the segmentation mini-stats badge for the column header."""
    return Span(
        render_seg_mini_stats_text(segments),
        id=CombinedHtmlIds.SEG_MINI_STATS,
        cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm),
        hx_swap_oob="true" if oob else None,
    )

In [ ]:
#| export
def render_align_mini_stats_badge(
    chunks:List[VADChunk],  # Current VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    """Render the alignment mini-stats badge for the column header."""
    return Span(
        render_align_mini_stats_text(chunks),
        id=CombinedHtmlIds.ALIGNMENT_MINI_STATS,
        cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm),
        hx_swap_oob="true" if oob else None,
    )

## Alignment Status Indicator

Shows whether segment count matches VAD chunk count for 1:1 alignment.
Displays different states: waiting for data, mismatch with delta, or aligned.

In [ ]:
#| export
def render_alignment_status_text(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> str:  # Status message text
    """Generate alignment status message based on segment and VAD chunk counts."""
    if segment_count == 0 or chunk_count == 0:
        return "Waiting for data..."
    
    delta = segment_count - chunk_count
    
    if delta == 0:
        return f"Aligned ({segment_count})"
    elif delta > 0:
        return f"{segment_count} segments vs {chunk_count} VAD chunks (merge {delta})"
    else:
        return f"{segment_count} segments vs {chunk_count} VAD chunks (split {-delta})"


#| export
def render_alignment_status(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Alignment status badge component
    """Render the alignment status indicator badge."""
    is_aligned = segment_count > 0 and chunk_count > 0 and segment_count == chunk_count
    
    # Choose badge style based on alignment state
    if is_aligned:
        badge_cls = combine_classes(badge, badge_sizes.sm, text_dui.success, font_weight.bold)
    elif segment_count == 0 or chunk_count == 0:
        badge_cls = combine_classes(badge, badge_styles.ghost, badge_sizes.sm)
    else:
        badge_cls = combine_classes(badge, badge_sizes.sm, text_dui.warning, font_weight.bold)
    
    return Span(
        render_alignment_status_text(segment_count, chunk_count),
        id=CombinedHtmlIds.ALIGNMENT_STATUS,
        cls=badge_cls,
        hx_swap_oob="true" if oob else None,
    )


# Shared footer inner content styling — flex-wrap allows vertical stacking on narrow viewports
_FOOTER_INNER_CLS = combine_classes(flex_display, flex_wrap.wrap, justify.between, items.center, w.full, gap(4))


#| export
def render_footer_inner_content(
    seg_footer:Any,  # Segmentation footer content (or None if not initialized)
    align_footer:Any,  # Alignment footer content (or None if not initialized)
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> Any:  # Styled wrapper div with both column footers and alignment status
    """Render the footer inner content with both column footers always present.
    
    Both footers remain in the DOM so their internal OOB targets (progress
    indicators, source position) are always valid regardless of which column
    is active. This enables cross-zone features like auto-play.
    """
    return Div(
        seg_footer or Span(),
        align_footer or Span(),
        render_alignment_status(segment_count, chunk_count),
        cls=_FOOTER_INNER_CLS
    )

## Shared Chrome Containers

Stable-ID containers for keyboard hints, toolbar, controls, and footer.
When decomposition is initialized, these are populated with real content.
Otherwise, placeholder text is shown. OOB innerHTML swaps update these
when focus switches between columns or after init completes.

In [ ]:
#| export
def _placeholder(
    text:str,  # Placeholder message
) -> Any:  # Styled placeholder paragraph
    """Render a placeholder text element for uninitialized chrome containers."""
    return P(text, cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50)))


def _render_shared_chrome(
    seg_state:dict=None,  # Extracted segmentation state (None = show placeholders)
    align_state:dict=None,  # Extracted alignment state (None = no VAD data yet)
    urls:SegmentationUrls=None,  # Segmentation URL bundle (required when seg_state provided)
    extra_actions:tuple=(),  # Extra toolbar elements (FA controls, sync toggle, etc.)
    nltk_split_disabled:bool=False,  # Whether NLTK Split button is disabled
) -> tuple:  # (toolbar, controls, footer)
    """Render shared chrome containers, populated with segmentation content when initialized.
    
    Takes extracted state dicts from `extract_seg_state()` and `extract_alignment_state()`
    which contain deserialized TextSegment and VADChunk objects.
    """
    is_init = seg_state is not None and seg_state.get("is_initialized", False)

    # --- Toolbar ---
    if is_init and urls:
        segments = seg_state.get("segments", [])
        history = seg_state.get("history", [])
        visible_count = seg_state.get("visible_count", DEFAULT_VISIBLE_COUNT)
        toolbar_content = render_toolbar(
            reset_url=urls.reset,
            ai_split_url=urls.ai_split,
            undo_url=urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
            extra_actions=extra_actions,
            nltk_split_disabled=nltk_split_disabled,
        )
    else:
        toolbar_content = _placeholder("Toolbar actions will appear here based on the active column.")

    toolbar = Div(
        toolbar_content,
        id=CombinedHtmlIds.SHARED_TOOLBAR,
        cls=str(p(2))
    )

    # --- Controls ---
    if is_init:
        card_width = seg_state.get("card_width", DEFAULT_CARD_WIDTH)
        controls_content = render_width_slider(SEG_CS_CONFIG, SEG_CS_IDS, card_width=card_width)
    else:
        controls_content = _placeholder("Column-specific controls will appear here.")

    controls = Div(
        controls_content,
        id=CombinedHtmlIds.SHARED_CONTROLS,
        cls=str(p(2))
    )

    # --- Footer with both column footers + alignment status ---
    # Note: seg_state["segments"] and align_state["vad_chunks"] are already deserialized
    # objects from extract_seg_state/extract_alignment_state — don't call from_dict() again
    seg_segments = seg_state.get("segments", []) if seg_state else []
    segment_count = len(seg_segments)
    align_chunks = align_state.get("vad_chunks", []) if align_state else []
    chunk_count = len(align_chunks)
    
    # Seg footer
    if is_init and seg_segments:
        focused_index = seg_state.get("focused_index", 0)
        seg_footer = render_seg_footer_content(seg_segments, focused_index)
    else:
        seg_footer = None

    # Align footer
    is_align_init = align_state is not None and align_state.get("is_initialized", False)
    if is_align_init and align_chunks:
        align_focused = align_state.get("focused_index", 0)
        align_footer = render_align_footer_content(align_chunks, align_focused)
    else:
        align_footer = None

    footer_inner = render_footer_inner_content(seg_footer, align_footer, segment_count, chunk_count)

    footer = Div(
        footer_inner,
        id=CombinedHtmlIds.SHARED_FOOTER,
        cls=combine_classes(
            p(1), bg_dui.base_100,
            border_dui.base_300, border.t(),
            flex_display, justify.center, items.center
        )
    )

    return toolbar, controls, footer


## Column Containers

Left column (60%, text decomposition) and right column (40%, VAD alignment).
Each column has a stable HTML ID, a header with mini-stats, and a content area.
When initialized, columns render the card stack viewport.
Otherwise they show a loading state with auto-trigger for initialization.
Columns stack vertically below the `lg` breakpoint.

In [ ]:
#| export
# Shared column styling (reused by init handler for outerHTML swap)
_SEG_COLUMN_CLS = combine_classes(
    w.full, w('[60%]').lg,
    min_h(0),
    flex_display, flex_direction.col,
    bg_dui.base_100, border_dui.base_300, border(1),
    border_radius.box,
    overflow.hidden,
    transition.all, duration._200,
)


def _render_seg_column(
    is_active:bool=True,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Left column component
    """Render the left segmentation column."""
    active_cls = combine_classes(ring(1), ring_dui.primary) if is_active else str(opacity(70))

    # Content area: initialized viewport or loading + auto-trigger
    if column_body is not None:
        content = column_body
    else:
        content = Div(
            render_loading_state(SEG_CS_IDS, message="Initializing segments..."),
            # Auto-trigger initialization on load
            Div(
                hx_post=init_url,
                hx_trigger="load",
                hx_target=CombinedHtmlIds.as_selector(
                    CombinedHtmlIds.SEG_COLUMN_CONTENT
                ),
                hx_swap="outerHTML"
            ) if init_url else None,
            id=CombinedHtmlIds.SEG_COLUMN_CONTENT,
            cls=combine_classes(grow(), overflow.hidden, flex_display, flex_direction.col, p(4))
        )

    return Div(
        _render_column_header(
            title="Text Segmentation",
            stats_id=CombinedHtmlIds.SEG_MINI_STATS,
            header_id=CombinedHtmlIds.SEG_COLUMN_HEADER,
            initial_text=mini_stats_text,
        ),
        content,
        id=CombinedHtmlIds.SEG_COLUMN,
        cls=combine_classes(_SEG_COLUMN_CLS, active_cls)
    )

In [ ]:
#| export
# Shared alignment column styling (reused by init handler for outerHTML swap)
_ALIGNMENT_COLUMN_CLS = combine_classes(
    w.full, w('[40%]').lg,
    min_h(0),
    flex_display, flex_direction.col,
    bg_dui.base_100, border_dui.base_300, border(1),
    border_radius.box,
    overflow.hidden,
    transition.all, duration._200,
)


def _render_alignment_column(
    is_active:bool=False,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Right column component
    """Render the right alignment column."""
    active_cls = combine_classes(ring(1), ring_dui.secondary) if is_active else str(opacity(70))

    # Content area: initialized viewport or loading + auto-trigger
    if column_body is not None:
        content = column_body
    else:
        content = Div(
            render_loading_state(ALIGN_CS_IDS, message="Analyzing audio..."),
            # Auto-trigger initialization on load
            Div(
                hx_post=init_url,
                hx_trigger="load",
                hx_target=CombinedHtmlIds.as_selector(
                    CombinedHtmlIds.ALIGNMENT_COLUMN_CONTENT
                ),
                hx_swap="outerHTML"
            ) if init_url else None,
            id=CombinedHtmlIds.ALIGNMENT_COLUMN_CONTENT,
            cls=combine_classes(grow(), min_h(0), overflow.hidden, flex_display, flex_direction.col, p(4))
        )

    return Div(
        _render_column_header(
            title="VAD Alignment",
            stats_id=CombinedHtmlIds.ALIGNMENT_MINI_STATS,
            header_id=CombinedHtmlIds.ALIGNMENT_COLUMN_HEADER,
            initial_text=mini_stats_text,
        ),
        content,
        id=CombinedHtmlIds.ALIGNMENT_COLUMN,
        cls=combine_classes(_ALIGNMENT_COLUMN_CLS, active_cls)
    )

## Keyboard System Container

Stable container for keyboard navigation system (script, hidden inputs, action buttons).
This container lives outside both column bodies so it doesn't get destroyed/recreated
when columns are swapped. OOB innerHTML swaps update its content when the combined
KB system is built (after both columns are initialized in 4d).

In [ ]:
#| export
def _render_keyboard_system_container(
    kb_system:Any=None,  # Rendered keyboard system (None = empty container)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Div with id=KEYBOARD_SYSTEM containing KB elements
    """Render stable container for keyboard navigation system elements."""
    if DEBUG_COMBINED_RENDER:
        print(f"[COMBINED_RENDER] _render_keyboard_system_container called, kb_system={kb_system is not None}")

    if kb_system is not None:
        content = (kb_system.script, kb_system.hidden_inputs, kb_system.action_buttons)
    else:
        content = ()

    return Div(
        *content,
        id=CombinedHtmlIds.KEYBOARD_SYSTEM,
        hx_swap_oob="true" if oob else None,
    )

## Main Combined Step Renderer

Top-level step renderer for the merged Phase 2. Composes the shared chrome,
dual-column layout, and a hidden input tracking the active column.

In [ ]:
#| export
def render_combined_step(
    ctx:InteractionContext,  # Interaction context with state and data
    seg_urls:SegmentationUrls=None,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls=None,  # URL bundle for alignment routes
    switch_chrome_url:str="",  # URL for chrome switching route
    fa_available:bool=False,  # Whether forced alignment plugin is available
    jm_trigger:Any=None,  # Pre-rendered job monitor trigger element (or None)
    fa_toggle_url:str="",  # URL for forced alignment toggle route
    jm_overlay_el:Any=None,  # Job monitor overlay element (or placeholder)
    jm_modal_el:Any=None,  # Job monitor modal element (or placeholder)
    jm_kb_script_el:Any=None,  # Job monitor keyboard script placeholder (for OOB pause/resume)
) -> Any:  # FastHTML component with full dual-column layout
    """Render Phase 2: Combined Segment & Align step with dual-column layout.
    
    The dual-column container has position:relative for the job monitor overlay,
    and a stable ID (CombinedHtmlIds.COLUMNS) for overlay targeting.
    """
    from cjm_transcript_segment_align.components.handlers import (
        build_fa_extra_actions, segments_match_presplit,
    )

    if DEBUG_COMBINED_RENDER:
        print("[COMBINED_RENDER] render_combined_step called")

    seg_urls = seg_urls or SegmentationUrls()

    # Extract state using combined helpers
    seg_state = extract_seg_state(ctx)
    align_state = extract_alignment_state(ctx)
    
    is_seg_init = seg_state["is_initialized"]
    is_align_init = align_state["is_initialized"]

    if DEBUG_COMBINED_RENDER:
        print(f"[COMBINED_RENDER] is_seg_init: {is_seg_init}")
        print(f"[COMBINED_RENDER] is_align_init: {is_align_init}")

    # Build FA extra_actions and nltk_split_disabled from state
    fa_extra = None
    nltk_disabled = False
    if is_seg_init:
        fa_extra = build_fa_extra_actions(
            seg_state, jm_trigger, fa_toggle_url, fa_available,
        )
        nltk_presplit = seg_state.get("nltk_presplit", [])
        nltk_disabled = segments_match_presplit(
            seg_state.get("segments", []), nltk_presplit,
        ) if nltk_presplit else True  # Disabled at init if no presplit saved yet

    # Build KB manager for modal (always — modal content is static)
    kb_manager = None
    if align_urls:
        kb_manager, _ = build_combined_kb_system(seg_urls, align_urls)

    # Variables for KB system and sync (only built when initialized)
    kb_system = None
    zone_change_js = None

    # Sync JS + should-play guard — generated unconditionally so they're available on first page load
    # (stacks auto-init via hx_trigger="load", these must already be in the DOM)
    sync_script = None
    should_play_script = None
    if align_urls:
        sync_script = generate_sync_script(
            source_input_id=SEG_CS_IDS.focused_index_input,
            target_nav_url=align_urls.card_stack.nav_to_index,
        )
        should_play_script = Script(generate_should_play_js())

    if is_seg_init:
        segments = seg_state["segments"]
        focused_index = seg_state["focused_index"]
        history = seg_state["history"]
        visible_count = seg_state["visible_count"]
        card_width = seg_state["card_width"]

        if align_urls:
            _, kb_system = build_combined_kb_system(seg_urls, align_urls)
            zone_change_js = generate_zone_change_js(switch_chrome_url)

        column_body = render_seg_column_body(
            segments=segments,
            focused_index=focused_index,
            visible_count=visible_count,
            card_width=card_width,
            urls=seg_urls,
            kb_system=None,
        )
        mini_stats_text = render_seg_mini_stats_text(segments)

        toolbar, controls, footer = _render_shared_chrome(
            seg_state=seg_state,
            align_state=align_state,
            urls=seg_urls,
            extra_actions=build_extra_actions(fa_extra),
            nltk_split_disabled=nltk_disabled,
        )

        seg_col = _render_seg_column(
            is_active=True,
            column_body=column_body,
            mini_stats_text=mini_stats_text,
        )
    else:
        toolbar, controls, footer = _render_shared_chrome(
            align_state=align_state,
        )
        seg_col = _render_seg_column(
            is_active=True,
            init_url=seg_urls.init,
        )

    # --- Alignment column ---
    if is_align_init and align_urls:
        chunks = align_state["vad_chunks"]
        align_focused = align_state["focused_index"]
        align_visible = align_state["visible_count"]
        align_width = align_state["card_width"]
        media_paths = align_state.get("media_paths", [])
        audio_src_url = align_urls.audio_src if align_urls else ""
        audio_urls = [f"{audio_src_url}?path={mp}" for mp in media_paths] if audio_src_url and media_paths else []

        align_body = render_align_column_body(
            chunks=chunks,
            focused_index=align_focused,
            visible_count=align_visible,
            card_width=align_width,
            urls=align_urls,
            kb_system=None,
            audio_urls=audio_urls,
            should_play_fn=SHOULD_PLAY_FN,
        )
        align_mini_text = render_align_mini_stats_text(chunks)
        align_col = _render_alignment_column(
            is_active=False,
            column_body=align_body,
            mini_stats_text=align_mini_text,
        )
    elif align_urls and align_urls.init:
        align_col = _render_alignment_column(
            is_active=False,
            init_url=align_urls.init,
        )
    else:
        align_col = _render_alignment_column(is_active=False)

    # Hidden input tracking active column
    active_column_input = Input(
        type="hidden",
        id=CombinedHtmlIds.ACTIVE_COLUMN_INPUT,
        name="active_column",
        value="seg",
    )

    # Stable keyboard system container
    kb_container = _render_keyboard_system_container(kb_system=kb_system)

    # Hidden button for chrome swap HTMX trigger
    chrome_switch_btn = None
    if switch_chrome_url and align_urls:
        chrome_switch_btn = Button(
            id=SWITCH_CHROME_BTN_ID,
            cls=str(display_tw.hidden),
            hx_post=switch_chrome_url,
            hx_include=f"#{CombinedHtmlIds.ACTIVE_COLUMN_INPUT}",
            hx_swap="none",
        )

    # Keyboard hints modal (rendered once, static — no OOB updates needed)
    if kb_manager:
        hints_modal, hints_trigger, hints_script = render_keyboard_hints_modal(kb_manager)
    else:
        hints_modal, hints_trigger, hints_script = Div(), Div(), Div()

    # Dual-column container with position:relative for job monitor overlay
    columns_children = [seg_col, align_col]
    if jm_overlay_el is not None:
        columns_children.append(jm_overlay_el)

    return Div(
        Div(
            Div(
                H2("Segment & Align", cls=combine_classes(font_size._3xl, font_weight.bold)),
                P(
                    "Decompose text into segments and align with audio timestamps.",
                    cls=combine_classes(text_dui.base_content.opacity(70), m.b(2))
                ),
            ),
            hints_trigger,
            cls=combine_classes(flex_display, items.start, justify.between),
        ),
        toolbar,
        controls,
        Div(
            *columns_children,
            id=CombinedHtmlIds.COLUMNS,
            cls=combine_classes(
                position.relative,  # For job monitor overlay positioning
                grow(), min_h(0),
                flex_display, flex_direction.col, flex_direction.row.lg,
                gap(4), overflow.hidden, p(1),
            )
        ),
        footer,
        kb_container,
        zone_change_js,
        sync_script,
        should_play_script,
        chrome_switch_btn,
        active_column_input,
        jm_modal_el,  # Job monitor modal (page-level, outside columns)
        jm_kb_script_el,  # Job monitor keyboard script placeholder (for OOB pause/resume)
        hints_modal,  # Keyboard hints modal dialog
        hints_script,  # Global ? key listener
        id=SegmentationHtmlIds.SEG_CONTAINER,
        cls=combine_classes(
            w.full, h.full,
            flex_display, flex_direction.col,
            p(4), p.x(2), p.b(0)
        )
    )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()